In [1]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained(
    "Qwen/Qwen3.5-9b",
    trust_remote_code=True,
)


/workspace/VLM2Vec/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import sys

sys.path.append('/workspace/VLM2Vec')

from src.arguments import ModelArguments, DataArguments
from src.model.model import MMEBModel
from src.model.processor import load_processor, QWEN3_5, VLM_IMAGE_TOKENS, Qwen3_5_process_fn
from src.utils.basic_utils import batch_to_device
from PIL import Image
import torch

model_args = ModelArguments(
    model_name='Qwen/Qwen3.5-9b',
    checkpoint_path='TIGER-Lab/VLM2Vec-Qwen3.5-9b',
    pooling='last',
    normalize=True,
    model_backbone='qwen3_5',
    lora=True
)
data_args = DataArguments()

processor = load_processor(model_args, data_args)
model = MMEBModel.load(model_args)
model = model.to('cuda', dtype=torch.bfloat16)
model.eval()

/workspace/VLM2Vec/.venv/lib/python3.12/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


FusedMLP of flash_attn is not installed!!!
DropoutAddRMSNorm of flash_attn is not installed!!!


ModuleNotFoundError: No module named 'flash_attn'

In [ ]:
# Image + Text -> Text
inputs = processor(text=f'{VLM_IMAGE_TOKENS[QWEN2_VL]} Represent the given image with the following question: What is in the image',
                   images=Image.open('../../../assets/example.jpg'),
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
inputs['pixel_values'] = inputs['pixel_values'].unsqueeze(0)
inputs['image_grid_thw'] = inputs['image_grid_thw'].unsqueeze(0)
qry_output = model(qry=inputs)["qry_reps"]

string = 'A cat and a dog'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))
## A cat and a dog = tensor([[0.3281]], device='cuda:0', dtype=torch.bfloat16)

string = 'A cat and a tiger'
inputs = processor(text=string,
                   images=None,
                   return_tensors="pt")
inputs = {key: value.to('cuda') for key, value in inputs.items()}
tgt_output = model(tgt=inputs)["tgt_reps"]
print(string, '=', model.compute_similarity(qry_output, tgt_output))
## A cat and a tiger = tensor([[0.2871]], device='cuda:0', dtype=torch.bfloat16)


# Batch processing
processor_inputs = {
    "text": [f'{VLM_IMAGE_TOKENS[QWEN2_VL]} Represent the given image with the following question: What is in the image',
          f'{VLM_IMAGE_TOKENS[QWEN2_VL]} Represent the given image with the following question: What is in the image'],
    "images": [Image.open('../../../assets/example.jpg'),
            Image.open('../../../assets/example.jpg')],
}
inputs = Qwen2_VL_process_fn(
    processor_inputs,
    processor)
inputs = batch_to_device(inputs, "cuda")
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    qry_output = model(qry=inputs)["qry_reps"]

processor_inputs = {
    "text": ['A cat and a dog', 'A cat and a tiger'],
    "images": [None, None],
}
inputs = Qwen2_VL_process_fn(
    processor_inputs,
    processor)
inputs = batch_to_device(inputs, "cuda")
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    tgt_output = model(tgt=inputs)["tgt_reps"]
print(model.compute_similarity(qry_output, tgt_output))
# tensor([[0.3316, 0.2900],
#         [0.3286, 0.2879]], device='cuda:0', grad_fn=<MmBackward0>)
